In [1]:
from pathlib import Path

import cantera
import polars
import util
from util.sim import reactors

tag = util.notebook_prefix()

In [2]:
DATA_PATH = Path("../data")

In [3]:
# Read in data and rename species to match simulation
species_df = polars.read_csv(DATA_PATH / "webb" / "species.csv")
species_dct = dict(species_df["concentration", "name"].drop_nulls().iter_rows())
conc_df = polars.read_csv(DATA_PATH / "webb" / "concentration.csv")
conc_df = conc_df.rename(species_dct, strict=False)

In [4]:
model = cantera.Solution(DATA_PATH / "cantera" / f"full_{tag}_calc.yaml")
concs = conc_df.select("CPT(563)", "N2", "O2(6)").rows(named=True)

In [5]:
temp = 825
pres = 1.1 * cantera.one_atm  # in atm.
tau = 4  # s
vol = 1 * (1e-2) ** 3  # m3

In [ ]:
# Run simulations for each point and store the results in an array
solns = cantera.SolutionArray(model)
for conc in concs:
    print(f"Starting simulation for {conc}")
    reactor = reactors.jsr(model=model, temp=temp, pres=pres, tau=tau, vol=vol, conc=conc)
    solns.append(reactor.thermo.state)

Starting simulation for {'CPT(563)': 0.005, 'N2': 0.9825, 'O2(6)': 0.0125}
Starting simulation for {'CPT(563)': 0.005, 'N2': 0.97625, 'O2(6)': 0.01875}
Starting simulation for {'CPT(563)': 0.005, 'N2': 0.973571429, 'O2(6)': 0.021428571}
Starting simulation for {'CPT(563)': 0.005, 'N2': 0.97, 'O2(6)': 0.025}
Starting simulation for {'CPT(563)': 0.005, 'N2': 0.965, 'O2(6)': 0.03}
Starting simulation for {'CPT(563)': 0.005, 'N2': 0.960909091, 'O2(6)': 0.034090909}
Starting simulation for {'CPT(563)': 0.005, 'N2': 0.9575, 'O2(6)': 0.0375}
Starting simulation for {'CPT(563)': 0.005, 'N2': 0.953333333, 'O2(6)': 0.041666667}
Starting simulation for {'CPT(563)': 0.005, 'N2': 0.945, 'O2(6)': 0.05}
Starting simulation for {'CPT(563)': 0.005, 'N2': 0.937307692, 'O2(6)': 0.057692308}
Starting simulation for {'CPT(563)': 0.005, 'N2': 0.9325, 'O2(6)': 0.0625}
Starting simulation for {'CPT(563)': 0.005, 'N2': 0.926818182, 'O2(6)': 0.068181818}
Starting simulation for {'CPT(563)': 0.005, 'N2': 0.92, '

In [12]:
sim_df = conc_df.with_columns(
    *(polars.Series(s, solns(s).X.flatten() * 10**6) for s in species_df["name"])
)
sim_df.write_csv(DATA_PATH / "cantera" / f"full_{tag}_calc.csv")
sim_df

phi,O2_molecules,CPT(563),N2,O2(6),H2(2),H2O(5),CO(12),CO2(13),CH4(33),CH3CHO(41),C2H4(52),C3H6(131),C3H4O(165),C4H6(227),C4H8(253),C5H6(478),C5H8(522),C5H8O(825)rs,C6H6(970),HO2(8),OH(4)
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
3.0,0.12233,0.002302,0.978504,0.008648,0.000738,0.003193,0.002246,0.000256,0.000273,0.000038,0.001565,0.000125,0.000357,0.000022,0.000043,0.000057,0.000578,0.000018,0.000002,0.000002,1.0116e-9
2.0,0.1835,0.001508,0.970308,0.012082,0.000954,0.005328,0.004198,0.000605,0.000381,0.00005,0.001882,0.000115,0.000333,0.000023,0.000035,0.000055,0.000578,0.000018,0.000003,0.000004,2.0968e-9
1.75,0.20962,0.001362,0.967274,0.014018,0.000969,0.005897,0.00464,0.000725,0.000383,0.00005,0.001886,0.000105,0.000314,0.000022,0.00003,0.000055,0.000581,0.000018,0.000003,0.000004,2.4419e-9
1.5,0.24456,0.001247,0.963499,0.016924,0.000951,0.006415,0.004953,0.000849,0.000369,0.000048,0.001849,0.000093,0.00029,0.000021,0.000025,0.000057,0.000594,0.000019,0.000003,0.000005,2.7735e-9
1.25,0.29347,0.001164,0.958489,0.021388,0.000892,0.006845,0.005086,0.000966,0.000341,0.000043,0.001768,0.00008,0.000262,0.000021,0.00002,0.000061,0.000621,0.00002,0.000002,0.000005,3.0590e-9
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
0.17,2.15785,0.00104,0.771172,0.211573,0.00021,0.007405,0.003119,0.000708,0.000073,0.00001,0.000589,0.000015,0.000088,0.000018,0.000002,0.000137,0.000942,0.000033,6.9418e-7,0.000007,3.5797e-9
0.15,2.44557,0.001038,0.741981,0.240974,0.00019,0.007383,0.003015,0.000666,0.000064,0.00001,0.000536,0.000014,0.000083,0.000018,0.000002,0.00014,0.00095,0.000034,6.2841e-7,0.000007,3.5859e-9
0.13,2.75147,0.001037,0.703777,0.279408,0.000168,0.007356,0.002902,0.000619,0.000055,0.000009,0.000481,0.000012,0.000078,0.000018,0.000002,0.000143,0.000958,0.000034,5.5726e-7,0.000007,3.5923e-9
